In [ ]:
# imports go here in first cell
import os
import time
import shutil
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

NB_DIR = Path.cwd()
PF_DIR = NB_DIR.parent                     # people_fusion/
DETECTOR_DIR = PF_DIR / "detector"
WEIGHTS_DIR = DETECTOR_DIR / "weights"
SHOTS_DIR = PF_DIR / "screenshots"


def show(img_bgr, title="", figsize=(11, 7)):
    """Display a BGR (OpenCV) image inline as RGB."""
    plt.figure(figsize=figsize)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis("off")
    plt.show()


def build_rtsp_url(host="10.0.0.24", user="admin", password=None, subtype="0"):
    """Build RTSP URL for Amcrest camera (matches detect.py logic)."""
    if password is None:
        password = os.environ.get("RTSP_PASSWORD", "")
    auth = f"{user}:{password}@" if password else f"{user}@"
    return f"rtsp://{auth}{host}:554/cam/realmonitor?channel=1&subtype={subtype}"


def open_capture(url):
    """Open RTSP stream with tcp transport and minimal buffering."""
    os.environ.setdefault("OPENCV_FFMPEG_CAPTURE_OPTIONS", "rtsp_transport;tcp")
    cap = cv2.VideoCapture(url, cv2.CAP_FFMPEG)
    cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
    return cap


print("cwd          :", NB_DIR)
print("weights dir  :", WEIGHTS_DIR)
print("screenshots  :", SHOTS_DIR)

In [ ]:
# take a screenshot with camera, save it in a subfolder and in /notebooks and display the pic below
# Needs the camera password in the env (same as the detector): export RTSP_PASSWORD='...'
url = build_rtsp_url()
cap = open_capture(url)
if not cap.isOpened():
    raise RuntimeError(f"cannot open RTSP stream: {url.replace(os.environ.get('RTSP_PASSWORD', ''), '***')}")

# Drain a few buffered frames so we grab a fresh, fully-decoded image.
frame = None
for _ in range(5):
    ok, frame = cap.read()
    if not ok or frame is None:
        time.sleep(0.1)
cap.release()
if frame is None:
    raise RuntimeError("failed to read a frame from the camera")

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"shot_{stamp}.png"

# Canonical copy in people_fusion/screenshots/, plus a convenience copy next to the notebook.
SHOTS_DIR.mkdir(parents=True, exist_ok=True)
shot_path = SHOTS_DIR / fname
nb_path = NB_DIR / fname
cv2.imwrite(str(shot_path), frame)
shutil.copyfile(shot_path, nb_path)

print(f"saved {frame.shape[1]}x{frame.shape[0]} -> {shot_path}")
print(f"copied -> {nb_path}")
show(frame, f"camera screenshot {stamp}")

In [ ]:
# prepare yolo model and print a summary of important params set
import torch

MODEL_PATH = WEIGHTS_DIR / "yolo11n.pt"   # 80-class COCO detector
CONF = 0.25
IMGSZ = 1280                               # matches detect.py default for the fisheye frame
DEVICE = "0" if torch.cuda.is_available() else "cpu"

model = YOLO(str(MODEL_PATH))

print("model      :", MODEL_PATH.name)
print("task       :", model.task)
print("num classes:", len(model.names))
print("conf thresh:", CONF)
print("imgsz      :", IMGSZ)
print("device     :", DEVICE, "->", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("classes    :", ", ".join(list(model.names.values())[:15]), "...")

In [ ]:
# preprocess screenshot and show result
# Ultralytics does its own letterbox/normalize internally, so we keep this light:
# work on a contiguous BGR copy and lift contrast a touch for the indoor scene.
img = frame.copy()
lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
l, a, b = cv2.split(lab)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
l = clahe.apply(l)
img = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)
img = np.ascontiguousarray(img)

print("preprocessed:", img.shape, img.dtype)
show(img, "preprocessed (CLAHE contrast)")

In [ ]:
# feed yolo the screenshot and output status
t0 = time.time()
result = model.predict(img, imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]
dt = time.time() - t0

n = 0 if result.boxes is None else len(result.boxes)
print(f"inference: {dt * 1000:.0f} ms | {n} detections")

counts = {}
if n:
    cls = result.boxes.cls.cpu().numpy().astype(int)
    conf = result.boxes.conf.cpu().numpy()
    for c, cf in zip(cls, conf):
        counts[result.names[c]] = counts.get(result.names[c], 0) + 1
    for label, k in sorted(counts.items(), key=lambda x: -x[1]):
        print(f"  {label:<15} x{k}")
else:
    print("  (nothing above conf threshold)")

In [ ]:
# show processed screenshot
overlay = result.plot()  # BGR image with boxes + labels drawn

out_path = SHOTS_DIR / f"shot_{stamp}_detected.png"
cv2.imwrite(str(out_path), overlay)
shutil.copyfile(out_path, NB_DIR / out_path.name)
print("saved overlay ->", out_path)
show(overlay, f"detections: {n} objects {counts}")